Cargar datos

In [48]:
import pandas as pd
import numpy as np
from typing import Union
import random

In [2]:
# Pruebas con el 50K
interactions = pd.read_csv("Datos/Pixel50k/interaction.csv")
item_info    = pd.read_csv("Datos/Pixel50k/item_info.csv")

In [4]:
# Párametros generales
N = 10      # tamaño recomendación
np.random.seed(5)       # semilla

In [9]:
# Separar train-test
# del práctico 8, 
def split_users(unique_uid: pd.Index, test_users_size: Union[float, int]) :

    n_users = unique_uid.size

    if isinstance(test_users_size, int):
        n_heldout_users = test_users_size
    else:
        n_heldout_users = int(test_users_size * n_users)

    tr_users = unique_uid[: (n_users - n_heldout_users * 2)]
    vd_users = unique_uid[(n_users - n_heldout_users * 2) : (n_users - n_heldout_users)]
    te_users = unique_uid[(n_users - n_heldout_users) :]

    return tr_users, vd_users, te_users

unique_uid = interactions.index
idx_perm = np.random.permutation(unique_uid.size)
unique_uid = unique_uid[idx_perm]
tr_users, vd_users, te_users = split_users(unique_uid, test_users_size=0.1)

In [17]:
train_df = interactions.loc[tr_users]
val_df   = interactions.loc[vd_users]
test_df  = interactions.loc[te_users]

random

In [25]:
# Creo que en esto no hay que incluir los datos de validación, pero no estoy seguro
def recomendaciones_random(train_df, test_df, N=10):
    usuarios_test = test_df['user_id'].unique()
    
    # Todos los ítems únicos disponibles en el catálogo (entrenamiento)
    todos_los_items = set(train_df["item_id"].unique())
    
    y_true = []
    y_pred = []

    for user_id in usuarios_test:
        vistos_train = set(train_df[train_df['user_id'] == user_id]['item_id'])
        vistos_test = test_df[test_df['user_id'] == user_id]['item_id'].head(N).tolist()    # Esto quizá debería calcularse afuera, como un ground_truth
        
        candidatos = list(todos_los_items - vistos_train)
        random_items = random.sample(candidatos, N)

        y_true.append(vistos_test)
        y_pred.append(random_items)
        
    return y_true, y_pred

In [26]:
y_true_random, y_pred_random = recomendaciones_random(train_df, test_df, N)

Most Popular

quiza debería pasar y_true como argumento a la función para no tener que hacer el cálculo todas las veces

In [42]:
def recomendaciones_most_popular(train_df, test_df, N):
    usuarios_test = test_df['user_id'].unique()

    y_true = []
    y_pred = []

    items_mas_populares = train_df.groupby('item_id')['user_id'].count().sort_values(ascending=False).index.tolist()

    for user_id in usuarios_test:
        vistos_train = set(train_df[train_df['user_id'] == user_id]['item_id'])
        vistos_test = test_df[test_df['user_id'] == user_id]['item_id'].tolist()    

        recomendaciones = [it for it in items_mas_populares if it not in vistos_test][:N]

        y_true.append(vistos_test)
        y_pred.append(recomendaciones)

    return y_true, y_pred

In [43]:
y_true_mostPop, y_pred_mostPop = recomendaciones_most_popular(train_df, test_df, N)

Evaluación métricas

In [ ]:
# De práctico_métricas.ipynb
def precision_at_k(r, k):
    assert 1 <= k <= r.size
    return (np.asarray(r)[:k] != 0).mean()

def average_precision_at_k(r, k):
    r = np.asarray(r)
    score = 0.
    for i in range(min(k, r.size)):
        score += precision_at_k(r, i + 1)
    return score / k

def dcg_at_k(r, k):
    r = np.asfarray(r)[:k]
    if r.size:
        return np.sum(np.subtract(np.power(2, r), 1) / np.log2(np.arange(2, r.size + 2)))
    return 0.

def idcg_at_k(k):
    return dcg_at_k(np.ones(k), k)

def ndcg_at_k(r, k, max_relevant):
    idcg = idcg_at_k(min(k, max_relevant))
    if not idcg:
        return 0.
    return dcg_at_k(r, k) / idcg

def recall_at_k(relevant_items, recommended_items, k):
    relevant_items = set(relevant_items)
    recommended_items = set(recommended_items[:k])
    intersection = relevant_items.intersection(recommended_items)
    recall = len(intersection) / len(relevant_items) if len(relevant_items) > 0 else 0
    return recall

def evaluate_model(y_true, y_pred, N):
    map_scores = 0.
    ndcg_scores = 0.
    recall_scores = 0.
    
    for true_items, pred_items in zip(y_true, y_pred):
        
        true_set = set(true_items)
        
        # el relevance vector
        r = [1 if item in true_set else 0 for item in pred_items]
        
        # MAP
        user_map = average_precision_at_k(r, N)
        map_scores += user_map
        # NDGC@N
        max_relevant = len(true_items)
        user_ndcg = ndcg_at_k(r, N, max_relevant)
        ndcg_scores += user_ndcg
        # Recall@N
        user_recall = recall_at_k(true_items, pred_items, N)
        recall_scores += user_recall
        
    cantidad = len(y_true)
    final_map = map_scores / cantidad
    final_ndcg = ndcg_scores / cantidad
    final_recall = recall_scores / cantidad
    
    print(f"Métricas de evaluación ranking (Top-{N}):")
    print(f"MAP@{N}:    {final_map:.4f}")
    print(f"nDCG@{N}:   {final_ndcg:.4f}")
    print(f"Recall@{N}: {final_recall:.4f}")
    
    return final_map, final_ndcg, final_recall


In [50]:
evaluate_model(y_true_random, y_pred_random, N)

AttributeError: `np.asfarray` was removed in the NumPy 2.0 release. Use `np.asarray` with a proper dtype instead.